# Coqui XTTS-v2 Voice Cloning in Google Colab

This notebook allows you to use Coqui XTTS-v2 for text-to-speech with voice cloning using a reference audio file.

## 2. Imports and Device Setup

In [1]:
from TTS.api import TTS
import torch
import os

# --- User Configuration for Device ---
# Set your desired device: "cuda" for GPU (if available), or "cpu".
TARGET_DEVICE = "mps"  # Options: "cuda" or "cpu" or "mps"
# -------------------------------------

# Determine the device to use
print(f"Target device specified: {TARGET_DEVICE}")
cuda_available = torch.backends.mps.is_available()
print(f"MPS available: {cuda_available}")

if TARGET_DEVICE == "cuda" and cuda_available:
    device = "cuda"
elif TARGET_DEVICE == "mps" and torch.backends.mps.is_available():
    device = "mps"
    print("MPS available: {torch.backends.mps.is_available()}")
elif TARGET_DEVICE == "cuda" and not cuda_available:
    device = "cpu"
    print("CUDA was targeted but is not available. Falling back to CPU.")
else:
    device = "cpu"

print(f"Using device: {device}")

from TTS.tts.models.xtts import GPT2InferenceModel  # Should work after clean install

Target device specified: mps
MPS available: True
MPS available: {torch.backends.mps.is_available()}
Using device: mps


/Users/ivkrasovskii/model-voice-generator/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'GPT2InferenceModel' from 'TTS.tts.models.xtts' (/Users/ivkrasovskii/model-voice-generator/.venv/lib/python3.11/site-packages/TTS/tts/models/xtts.py)

### Specify Text to Synthesize and Language

In [2]:
# Edit this line to change the text you want to synthesize.
text_to_speak = "Hello, this is a test of your custom voice workflow in Google Colab."

# Set the language for XTTS (e.g., "en", "es", "fr", "de", etc.)
language_to_use = "en"

output_wav_path = "output_trimmed.wav"

print(f"Text to synthesize: {text_to_speak}")
print(f"Language: {language_to_use}")
print(f"Output file will be: {output_wav_path}")

Text to synthesize: Hello, this is a test of your custom voice workflow in Google Colab.
Language: en
Output file will be: output_trimmed.wav


## 4. Initialize TTS Model and Perform Inference

In [10]:
import os
import torch

# Alternative fix: Patch torch.load to use weights_only=False
original_load = torch.load

def patched_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

torch.load = patched_load
print("Applied torch.load patch for TTS compatibility")

from TTS.api import TTS

# Fix for GPT2InferenceModel generate method - avoid recursion
import types
try:
    from TTS.tts.models.xtts import GPT2InferenceModel
    
    # Store original __getattr__ if it exists
    original_getattr = getattr(GPT2InferenceModel, '__getattr__', None)
    
    def patched_getattr(self, name):
        if name == 'generate' and hasattr(self, 'model'):
            return self.model.generate
        elif original_getattr:
            return original_getattr(self, name)
        else:
            raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")
    
    GPT2InferenceModel.__getattr__ = patched_getattr
    print("Applied GPT2InferenceModel __getattr__ patch")
except ImportError as e:
    print(f"GPT2InferenceModel patch not needed or failed: {e}")

# Your existing variables
reference_wav_path = "output_trimmed.wav"
output_wav_path = "output_trimmed_2.wav"
text_to_speak = "Hello, this is a test of your custom voice workflow in Google Colab."
language_to_use = "en"
device = "cpu"  # or "cuda" if you have GPU

if reference_wav_path and os.path.exists(reference_wav_path):
    try:
        print(f"Initializing TTS model on device: {device}...")
        # Model name for XTTS-v2
        model_name = "tts_models/multilingual/multi-dataset/xtts_v2"
        tts = TTS(model_name).to(device)
        
        # Apply the generate patch to the loaded model instance
        if hasattr(tts.synthesizer.tts_model, 'gpt') and hasattr(tts.synthesizer.tts_model.gpt, 'model'):
            gpt_inference = tts.synthesizer.tts_model.gpt
            if not hasattr(gpt_inference, 'generate'):
                def generate_method(self, *args, **kwargs):
                    return self.model.generate(*args, **kwargs)
                gpt_inference.generate = types.MethodType(generate_method, gpt_inference)
                print("Applied instance-level generate patch")
        
        print("TTS model initialized successfully.")

        print(f"Generating speech for text: \"{text_to_speak}\"")
        print(f"Using reference audio: {reference_wav_path}")
        tts.tts_to_file(
            text=text_to_speak,
            speaker_wav=reference_wav_path,
            language=language_to_use,
            file_path=output_wav_path
        )
        print(f"XTTS speech saved to {output_wav_path}")
        print("You can now download the file in the next step.")

    except Exception as e:
        print(f"An error occurred during TTS processing: {e}")
        if device == "cuda":
            print("If this is a CUDA-related error (e.g., 'CUDA out of memory'), try restarting the runtime and selecting 'cpu' for TARGET_DEVICE in Cell 2.")
else:
    print("Reference WAV path is not set or file does not exist. Please upload a reference audio file in Step 3.")

Applied torch.load patch for TTS compatibility
GPT2InferenceModel patch not needed or failed: cannot import name 'GPT2InferenceModel' from 'TTS.tts.models.xtts' (/Users/ivkrasovskii/model-voice-generator/.venv/lib/python3.11/site-packages/TTS/tts/models/xtts.py)
Initializing TTS model on device: cpu...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
An error occurred during TTS processing: maximum recursion depth exceeded


## 5. Download the Generated Speech

In [ ]:
if os.path.exists(output_wav_path):
  from google.colab import files
  files.download(output_wav_path)
else:
  print(f"Output file {output_wav_path} not found. Please ensure the previous cell ran successfully.")